In [2]:
from wekeo.download import get_FRP_products
from wekeo.reader import read_FRP_product
import xarray as xr

from core import log as log
from core import env, save
from datetime import date
    
def get_log_event(day: date) -> xr.Dataset:
    """Download and Compile multiple FRP files into a single xarray Dataset."""
    
    log.info(f"Downloading FRP products for date: {day}")
    
    eday = day.strftime("%Y-%m-%d")
    sday = day.strftime("%Y-%m-%d") # same day for single day query

    files = get_FRP_products(
        start_date = sday,
        end_date = eday,
    )

    # Read specific variables
    variables = [
        'latitude', 
        'longitude',
        'FRP_SWIR',
        'FRP_uncertainty_SWIR',
        'FRP_MWIR', 
        'FRP_uncertainty_MWIR',
        'confidence_SWIR_SAA',  # SAA flag for quality control
        'solar_zenith',
        # 'S8_Fire_pixel_BT',  # This is from a different dimension
    ]
    
    log.info(f"Compiling {len(files)} FRP products into a Log Event dataset")
    datasets = []
    
    for f in files:
        ds = read_FRP_product(f, variables=variables).compute()
        datasets.append(ds)
        
    log_event = xr.concat(datasets, dim='merged_MWIR1kmStandard_SWIR1km')
    log_event = log_event.rename_dims({"merged_MWIR1kmStandard_SWIR1km": "nb_detection"})
    log_event = log_event.rename_vars({"solar_zenith": "sza"})
    
    return log_event

day = date(2022, 7, 15)
log_event = get_log_event(day)
sday = day.strftime("%Y_%m_%d")
print(log_event)

output_dir = env.getdir("OUTPUT_DIR")
save.to_netcdf(log_event, f"{output_dir}/log_event_{sday}.nc")


17:24:08 (i) Downloading FRP products for date: 2022-07-15
17:24:13 (i) All files already present locally, skipping download.
17:24:13 (i) Compiling 193 FRP products into a Log Event dataset
/tmp/ipykernel_1954702/2981505431.py:42: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'fires_SWIR500m' ('fires_SWIR500m',) The recommendation is to set join explicitly for this case.
  log_event = xr.concat(datasets, dim='merged_MWIR1kmStandard_SWIR1km')
/tmp/ipykernel_1954702/2981505431.py:42: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinate

<xarray.Dataset> Size: 1MB
Dimensions:                         (nb_detection: 17959, fires_SWIR500m: 888,
                                     fires_MWIR1km_alternative: 5405)
Coordinates:
  * fires_SWIR500m                  (fires_SWIR500m) int64 7kB 0 1 2 ... 886 887
  * fires_MWIR1km_alternative       (fires_MWIR1km_alternative) int64 43kB 0 ...
  * merged_MWIR1kmStandard_SWIR1km  (nb_detection) int64 144kB 0 0 1 ... 52 53
Dimensions without coordinates: nb_detection
Data variables:
    FRP_SWIR                        (nb_detection) float64 144kB 1.824 ... -1.0
    confidence_SWIR_SAA             (nb_detection) int32 72kB 100 100 ... -100
    FRP_uncertainty_MWIR            (nb_detection) float64 144kB -1.0 ... 1.776
    sza                             (nb_detection) float64 144kB 146.8 ... 47.89
    FRP_uncertainty_SWIR            (nb_detection) float64 144kB 8.253 ... 0.0
    FRP_MWIR                        (nb_detection) float64 144kB -1.0 ... 6.816
    latitude                  

17:25:41 (d) Moving "/tmp/tmp_n77e_89/log_event_2022_07_15.nc" to "/mnt/ceph/proj/WEKEO/outputs/log_event_2022_07_15.nc"...


In [ ]:
import xarray as xr

f = "/mnt/ceph/proj/IASI_PCA3/PROD_TEST/METOP-C_IASI_PCA3-LogEvent-L1a.v1.00/2024/2024_07_15/METOP-C_IASI_PCA3-LogEvent-L1a_2024-07-15_V1-00.nc"
ds = xr.open_dataset(f)

print(ds)
for d in ds.data_vars:
    print(d, ds[d].attrs)

In [ ]:
from wekeo.log_event_accumulator import accumulate_events_to_grid

f = "/mnt/ceph/proj/WEKEO/outputs/log_event_2022_07_15.nc"
log_event = xr.open_dataset(f)

l3 = accumulate_events_to_grid(
    log_event,
    variables=["FRP_SWIR", "FRP_MWIR"],
    width=720,
    height=360,
)